<a href="https://colab.research.google.com/github/AaschitaReddyM/EcoPulse-Multi-Task-Spatial-CNN-for-Spatiotemporal-Environmental-Health-Forecasting/blob/main/EcoPulse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 1. Hardware Pipeline Initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"EcoPulse Deep Learning Core initializing on target: {device}")

# 2. EcoPulse Spatiotemporal Grid Data Ingestion Engine
class EcoPulseSpatialDataset(Dataset):
    def __init__(self, num_samples=128):
        self.num_samples = num_samples
        # Simulates a 3-channel spatial matrix grid (e.g., Channel 0: PM2.5, Channel 1: Ozone, Channel 2: Temp) [cite: 120]
        # Sized at 64x64 pixels/spatial regions to replicate localized Uber H3 cell groupings [cite: 145]
        self.spatial_exposure_grids = torch.randn(num_samples, 3, 64, 64)

        # Parallel Multi-Task target registers (1 = Critical Risk Event, 0 = Baseline Stable) [cite: 144]
        self.respiratory_labels = torch.randint(0, 2, (num_samples, 1)).float()
        self.cardiovascular_labels = torch.randint(0, 2, (num_samples, 1)).float()
        self.metabolic_labels = torch.randint(0, 2, (num_samples, 1)).float()

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return (self.spatial_exposure_grids[idx],
                self.respiratory_labels[idx],
                self.cardiovascular_labels[idx],
                self.metabolic_labels[idx])

# Instantiate batch workers for high-throughput memory streaming
train_loader = DataLoader(EcoPulseSpatialDataset(num_samples=256), batch_size=16, shuffle=True)

# 3. EcoPulse Multi-Task Spatial Convolutional Neural Network (CNN) Framework
class EcoPulseMultiTaskCNN(nn.Module):
    def __init__(self):
        super(EcoPulseMultiTaskCNN, self).__init__()

        # Shared Convolutional Block (Extracts features from overlapping spatial pollution/weather grids)
        self.shared_convolutional_core = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Dim reduction: 16 x 32 x 32

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) # Dim reduction: 32 x 16 x 16
        )
        self.flatten_features_dimension = 32 * 16 * 16

        # Segmented Predictive Heads (Explicitly mapped to EcoPulse Layer Product Spec) [cite: 144]
        self.respiratory_head = nn.Sequential(
            nn.Linear(self.flatten_features_dimension, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
        self.cardiovascular_head = nn.Sequential(
            nn.Linear(self.flatten_features_dimension, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
        self.metabolic_head = nn.Sequential(
            nn.Linear(self.flatten_features_dimension, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, spatial_tensor):
        # Extract features through shared 2D image-convolutions
        latent_features = self.shared_convolutional_core(spatial_tensor)
        flattened_vector = latent_features.view(latent_features.size(0), -1)

        # Diverge to isolated task predictions [cite: 144]
        airway_risk = self.respiratory_head(flattened_vector)
        cardio_risk = self.cardiovascular_head(flattened_vector)
        glycemic_risk = self.metabolic_head(flattened_vector)

        return airway_risk, cardio_risk, glycemic_risk

# 4. Model Instantiation & Mathematical Parameter Optimization Tuning
ecopulse_model = EcoPulseMultiTaskCNN().to(device)
loss_calculator = nn.BCELoss()
optimizer = optim.Adam(ecopulse_model.parameters(), lr=0.001)

# Clinical Priority Multipliers (Alpha, Beta, Gamma knobs from specification) [cite: 138]
alpha, beta, gamma = 1.0, 2.0, 1.0 # Emphasizing cardiovascular mistakes to prioritize high-risk events [cite: 139, 140]

# 5. Core Pipeline Backpropagation Training Loop
print("\nInitiating training loop for EcoPulse Multi-Task Environmental Spatial Engine...")
ecopulse_model.train()
for epoch in range(3):
    total_epoch_loss = 0.0
    for grids, resp_y, cardio_y, meta_y in train_loader:
        grids, resp_y = grids.to(device), resp_y.to(device)
        cardio_y, meta_y = cardio_y.to(device), meta_y.to(device)

        optimizer.zero_grad()

        # Forward Pass through network layers
        pred_resp, pred_cardio, pred_meta = ecopulse_model(grids)

        # Calculate isolated branch penalties [cite: 125]
        loss_r = loss_calculator(pred_resp, resp_y)
        loss_c = loss_calculator(pred_cardio, cardio_y)
        loss_m = loss_calculator(pred_meta, meta_y)

        # Master Composite Loss Formula Evaluation [cite: 125]
        consolidated_loss = (alpha * loss_r) + (beta * loss_c) + (gamma * loss_m)

        # Backpropagation
        consolidated_loss.backward()
        optimizer.step()
        total_epoch_loss += consolidated_loss.item()

    print(f"Epoch [{epoch+1}/3] - EcoPulse Consolidated Penalty: {total_epoch_loss/len(train_loader):.4f}")

# 6. Post-Training Optimization Strategy (Dynamic Quantization for Edge Systems) [cite: 154]
print("\nInitiating EcoPulse Edge Optimization Engine...")
ecopulse_model.to('cpu') # Quantization matrices execute on CPU tensors
quantized_edge_model = torch.quantization.quantize_dynamic(
    ecopulse_model, {nn.Linear}, dtype=torch.qint8
)
print("EcoPulse pipeline complete. Multi-task linear structures compressed dynamically from FP32 to signed INT8 configuration.")

EcoPulse Deep Learning Core initializing on target: cuda

Initiating training loop for EcoPulse Multi-Task Environmental Spatial Engine...
Epoch [1/3] - EcoPulse Consolidated Penalty: 4.4380
Epoch [2/3] - EcoPulse Consolidated Penalty: 2.7046
Epoch [3/3] - EcoPulse Consolidated Penalty: 2.2304

Initiating EcoPulse Edge Optimization Engine...
EcoPulse pipeline complete. Multi-task linear structures compressed dynamically from FP32 to signed INT8 configuration.


/tmp/ipykernel_608/798863038.py:121: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_edge_model = torch.quantization.quantize_dynamic(


In [3]:
import time

# Create a single synthetic spatial environmental image to simulate a real-time incoming reading
sample_input = torch.randn(1, 3, 64, 64)

print("⏱️ Initiating Real-Time Inference Latency Benchmark...")

# 1. Measure Baseline Model Latency (FP32)
ecopulse_model.to('cpu')  # Move to CPU to accurately simulate an embedded microprocessor environment
start_time = time.time()
for _ in range(100):  # Run 100 times to get a reliable average
    _ = ecopulse_model(sample_input)
baseline_latency = (time.time() - start_time) / 100 * 1000  # Convert to milliseconds

# 2. Measure Optimized Edge Model Latency (INT8)
start_time = time.time()
for _ in range(100):
    _ = quantized_edge_model(sample_input)
quantized_latency = (time.time() - start_time) / 100 * 1000

# 3. Compute Metrics
speedup_factor = baseline_latency / quantized_latency

print(f"\n================ BENCHMARK REPORT ================")
print(f"📊 Baseline Model Inference Latency (FP32): {baseline_latency:.3f} ms")
print(f"🚀 EcoPulse Edge Model Inference Latency (INT8): {quantized_latency:.3f} ms")
print(f"⚡ Edge Compute Acceleration Factor: {speedup_factor:.2f}x Faster")
print(f"==================================================")

⏱️ Initiating Real-Time Inference Latency Benchmark...

================ BENCHMARK REPORT ================
📊 Baseline Model Inference Latency (FP32): 4.373 ms
🚀 EcoPulse Edge Model Inference Latency (INT8): 3.586 ms
⚡ Edge Compute Acceleration Factor: 1.22x Faster
